# 1. What you'll learn

        - load a pretrained MobileNetV3 backbone
- compare pretrained and random initialization fairly
- break transferred features with a colour/channel shift

        **The one question this notebook answers:** Does a pretrained visual backbone learn faster than the same architecture starting from random weights under one small budget?

# 2. Setup

This lesson keeps transfer learning on a small real image set small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import os
import torch
from torch import nn
from torch.utils.data import DataLoader,Subset,TensorDataset
from torchvision import datasets,transforms,models
from sklearn.metrics import accuracy_score

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

A cached slice of real FashionMNIST supplies ten clothing classes, converted to RGB and resized for MobileNet. Reusing the verified image cache avoids a second 170 MB download while still testing transfer on real images.

**Small example:** If a pretrained filter already detects edges, five labelled images can teach a new head to combine those responses. A random filter must first discover edges and the class combination.

In [ ]:
torch.manual_seed(SEED);cache=Path.cwd().parent/"_shared_data";os.environ["TORCH_HOME"]=str(cache/"torch");tfm=transforms.Compose([transforms.Resize((64,64)),transforms.Grayscale(num_output_channels=3),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
try:
 full_train=datasets.FashionMNIST(cache,train=True,download=False,transform=tfm);full_test=datasets.FashionMNIST(cache,train=False,download=False,transform=tfm);train_ds=Subset(full_train,list(range(1000)));test_ds=Subset(full_test,list(range(400)));class_names=full_train.classes;source="real FashionMNIST transfer set"
except Exception:
 print("OFFLINE BANNER: real image cache absent; using generated colour textures.");rng=np.random.default_rng(SEED)
 def synth(n):
  lab=rng.integers(0,10,n);ims=rng.normal(0,.08,(n,3,64,64)).astype("float32")
  for j,c in enumerate(lab): ims[j,c%3,c*5:(c*5)+12,:]+=.8
  return TensorDataset(torch.tensor(ims),torch.tensor(lab))
 train_ds=synth(1000);test_ds=synth(400);class_names=[f"class {i}" for i in range(10)];source="synthetic fallback"
print("Source:",source,"train/test:",len(train_ds),len(test_ds),"sample:",train_ds[0][0].shape)

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
ims=torch.stack([train_ds[i][0] for i in range(12)]);labs=np.array([int(train_ds[i][1]) for i in range(len(train_ds))]);show=ims[:8].permute(0,2,3,1).numpy();show=(show-show.min())/(show.max()-show.min());fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].imshow(np.hstack(show));axes[0].axis("off");axes[0].set_title("Real clothing examples")
axes[1].bar(range(10),np.bincount(labs,minlength=10),color="#176b66");axes[1].set_title("Subset balance");axes[2].bar(["red","green","blue"],ims.mean((0,2,3)).numpy(),color=["#c95f5f","#5f9b78","#5f78b5"]);axes[2].set_title("Normalized channel means");plt.tight_layout();plt.show()

**Takeaway:** Objects are tiny and visually varied, making 1,000 labels a small budget.

**Takeaway:** The sequential slice is not perfectly balanced, so majority accuracy is reported.

**Takeaway:** Pretrained filters expect normalized RGB structure.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
gen=torch.Generator().manual_seed(SEED);train_loader=DataLoader(train_ds,batch_size=32,shuffle=True,generator=gen,num_workers=0);test_loader=DataLoader(test_ds,batch_size=64,shuffle=False,num_workers=0);print("One batch:",next(iter(train_loader))[0].shape)

# 6. Train

        Training turns the assumptions behind transfer learning on a small real image set into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Load ImageNet-pretrained convolutional features.
2. Freeze most backbone parameters so the small dataset trains a compact head.
3. Train the new classifier for the same batches as a random-backbone control.
4. Compare held-out accuracy under identical data and optimizer budgets.

In [ ]:
def make_model(pretrained):
 weights=models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
 try: m=models.mobilenet_v3_small(weights=weights)
 except Exception:
  print("WEIGHT BANNER: pretrained weights unavailable; comparison uses random initialization.");m=models.mobilenet_v3_small(weights=None);pretrained=False
 for p in m.features.parameters(): p.requires_grad=False
 m.classifier[3]=nn.Linear(m.classifier[3].in_features,10);return m,pretrained
def train_one(m):
 opt=torch.optim.Adam([p for p in m.parameters() if p.requires_grad],lr=.003);m.train();total=0
 for xb,yb in train_loader: opt.zero_grad();loss=nn.functional.cross_entropy(m(xb),yb);loss.backward();opt.step();total+=loss.item()*len(xb)
 return total/len(train_ds)
pre,used_pretrained=make_model(True);scratch,_=make_model(False);pre_loss=train_one(pre);scratch_loss=train_one(scratch);print("One-epoch losses:",round(pre_loss,3),round(scratch_loss,3),"pretrained used:",used_pretrained)

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
def predict(m,loader,swap=False):
 m.eval();a=[];b=[]
 with torch.no_grad():
  for x,y in loader: x=x[:,[2,1,0]] if swap else x;a.extend(y.numpy());b.extend(m(x).argmax(1).numpy())
 return np.array(a),np.array(b)
truth,pp=predict(pre,test_loader);_,sp=predict(scratch,test_loader);pa=accuracy_score(truth,pp);sa=accuracy_score(truth,sp);base=max(np.bincount(labs))/len(labs)
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].bar(["majority","pretrained","scratch"],[base,pa,sa],color=[".6","#176b66","#d38b45"]);axes[0].set_title("Same one-epoch budget")
axes[1].hist(torch.softmax(pre(next(iter(test_loader))[0]),1).max(1).values.detach().numpy(),bins=12,color="#176b66");axes[1].set_title("Pretrained-head confidence")
axes[2].bar(["pretrained loss","scratch loss"],[pre_loss,scratch_loss],color=["#176b66","#d38b45"]);axes[2].set_title("Training loss after budget");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print(f"Accuracy pretrained={pa:.3f}; scratch={sa:.3f}; majority={base:.3f}")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Transferred filters carry assumptions about RGB statistics and source-domain features; channel or domain shift can erase the advantage. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
_,swapped=predict(pre,test_loader,swap=True);print(f"Normal RGB accuracy={pa:.3f}; red/blue-swapped accuracy={accuracy_score(truth,swapped):.3f}; damage={pa-accuracy_score(truth,swapped):+.3f}")

**Use this when…** your labelled image set is small and the source pretraining domain shares useful visual primitives.

        **Don't use this when…** input channels or visual domain differ radically, weights cannot be redistributed, or scratch data are already abundant.

        ## Try this

        1. Unfreeze the last backbone block at a ten-times-smaller learning rate.
2. Use a stratified subset rather than the first 1,000 images.
3. Add crop/flip augmentation and rerun both budgets.